# KLA Image Restoration - Direct Upload Colab Training Pipeline (Aug 12, 2026)

This notebook allows you to train your model on Google Colab by uploading your code and dataset ZIPs directly through your browser, without connecting or mounting Google Drive.

### Setup Instructions:
1. Open this notebook in Google Colab.
2. Enable GPU: Go to **Runtime > Change runtime type > T4 GPU** (or A100/V100 if available).
3. Open the Files panel by clicking the **Folder icon (📁)** on the left sidebar.
4. Drag and drop **`i4c_project.zip`** and **`train.zip`** from your computer directly into that sidebar Files panel.
5. Once the uploads are complete, run the cells below sequentially.

## 1. Extract Archives
Unzips both the codebase and the training dataset into the local fast storage of the Colab instance.

In [ ]:
import os

project_zip = 'i4c_project.zip'
dataset_zip = 'train.zip'

print("Checking for uploaded ZIP files...")
if not os.path.exists(project_zip):
    raise FileNotFoundError("i4c_project.zip not found in Colab's Files panel! Please upload it first.")
if not os.path.exists(dataset_zip):
    raise FileNotFoundError("train.zip not found in Colab's Files panel! Please upload it first.")

print("Extracting project repository...")
!unzip -q {project_zip} -d i4c

print("Extracting dataset...")
!unzip -q {dataset_zip} -d dataset
print("Extraction complete!")

## 2. Configure Scaffolding & Directory Links
Creates symlinks inside `i4c/data/raw/` pointing to the training dataset so that path resolution matches your local repository structure.

In [ ]:
import os

# Create raw data directories inside i4c
os.makedirs('i4c/data/raw/train', exist_ok=True)

# Create links from the extracted dataset to i4c data paths
!ln -sfn /content/dataset/train/GT /content/i4c/data/raw/train/GT
!ln -sfn /content/dataset/train/NoisyLR /content/i4c/data/raw/train/NoisyLR

print("Scaffolding set up. Verification of directories:")
print("GT files found:", os.path.exists('i4c/data/raw/train/GT/000000.npy'))
print("NoisyLR files found:", os.path.exists('i4c/data/raw/train/NoisyLR/000000.npy'))

## 3. Install Dependencies
Installs necessary external metrics and configuration reading libraries.

In [ ]:
!pip install -q lpips torchmetrics pyyaml tqdm opencv-python matplotlib

## 4. Execute Training Loop
Launches the training loop. Checkpoints are automatically saved to `weights/` and metrics are logged to `outputs/training_log.csv`.

In [ ]:
%cd /content/i4c
%env PYTHONPATH=.

# Start PyTorch training
!python train.py

## 5. Download Weights & Metrics directly to Browser
Downloads the final trained checkpoint `model_best.pt` and the `training_log.csv` directly to your local computer's Downloads directory.

In [ ]:
from google.colab import files
import os

best_model_path = '/content/i4c/weights/model_best.pt'
log_path = '/content/i4c/outputs/training_log.csv'

print("Downloading best model weights...")
if os.path.exists(best_model_path):
    files.download(best_model_path)
else:
    print("Error: model_best.pt not found!")

print("Downloading training metrics log...")
if os.path.exists(log_path):
    files.download(log_path)
else:
    print("Error: training_log.csv not found!")